# Tugas 1 — Milestone 3: Analisis Temporal, Spasial, dan Advanced EDA

**Mata Kuliah:** Analisis Big Data  
**Target Check-in:** Pertemuan 8 (Mulai dikerjakan Pertemuan 6)  
**Nama Mahasiswa:** Novita Dian Nazarina  
**NIM:** 202310370311395    
**Username GitHub:** novittavn  

---

## Panduan Pengerjaan Milestone 3:
1. **Sumber Data:** Gunakan dataset yang telah dibersihkan pada Milestone 2 (`02_data_cleaning.ipynb`).
2. **Analisis Temporal / Windowing:** Terapkan minimal satu operasi temporal (Polars `group_by_dynamic()`, `rolling_mean()`, `shift()`, atau DuckDB window function `OVER (...)`).
3. **Visualisasi Interaktif:** Bangun minimal 6 visualisasi interaktif informatif menggunakan Plotly Express / Graph Objects atau Altair.
4. **Perumusan Insight:** Tuliskan minimal 5 temuan atau wawasan analitik mendalam berdasarkan hasil visualisasi data di atas.


## Bagian 1: Inisialisasi dan memuat data bersih

Jalankan notebook 02 terlebih dahulu, kemudian pilih **Restart Kernel and Run All**
pada notebook ini. Path mendukung folder kerja root proyek dan `notebooks/`.

Pertanyaan analisis: bagaimana tren penjualan, sebaran nilai pesanan, hubungan
ukuran keranjang dengan nilainya, kontribusi negara, dan pola hari–jam transaksi?

**Definisi metrik dan batas analisis:**
- Data asal berupa baris item; jumlah pesanan dihitung dengan invoice unik.
- Penjualan positif = `Quantity × Price_Original` dalam GBP untuk kuantitas dan
  harga positif serta invoice yang tidak diawali `C`. `Price` hasil winsorization
  pada notebook 02 tidak digunakan untuk menghitung nilai transaksi.
- Retur/pembatalan dilaporkan terpisah; total penjualan positif bukan pendapatan
  neto atau laba. ID pelanggan kosong tetap disertakan untuk analisis penjualan.
- Hari tanpa penjualan diisi nol selama rentang observasi. Hal ini tidak otomatis
  menunjukkan toko tutup. Bulan parsial dikeluarkan dari perbandingan bulan penuh.
- Jam mengikuti timestamp sumber tanpa konversi ke WIB karena zona waktu tidak
  tersedia. Negara merupakan lokasi pelanggan, bukan lokasi cabang toko.

Dataset yang telah dipilih pada notebook 01–02 adalah **Online Retail II dari
Inggris**, sehingga syarat dataset Indonesia pada README tugas masih belum terpenuhi.
Sumber: [Chen, D. (2012), UCI Online Retail II](https://archive.ics.uci.edu/dataset/502/online+retail+ii),
[DOI](https://doi.org/10.24432/C5CG6D), CC BY 4.0. UCI menjelaskan harga dalam
sterling dan awalan invoice `C` sebagai pembatalan.

In [ ]:
from pathlib import Path
import math
import duckdb
import polars as pl
import plotly
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
from IPython.display import Markdown, display

PROJECT_ROOT = next(
    (p for p in [Path.cwd().resolve(), *Path.cwd().resolve().parents]
     if (p / 'notebooks/02_data_cleaning.ipynb').is_file()), None
)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Jalankan dari root proyek atau folder notebooks/.')
DATA_PATH = PROJECT_ROOT / 'data/processed/dataset_bersih.parquet'
FIGURE_DIR = PROJECT_ROOT / 'output/figures'
TABLE_DIR = PROJECT_ROOT / 'output'
if not DATA_PATH.is_file():
    raise FileNotFoundError('Jalankan notebook 02 untuk membuat dataset_bersih.parquet.')
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
px.defaults.template = 'plotly_white'
px.defaults.color_discrete_sequence = px.colors.qualitative.Safe
pio.renderers.default = 'plotly_mimetype'
figures = {}

def tampilkan(fig, nama):
    fig.update_layout(font=dict(family='Arial', size=13), margin=dict(t=85, b=65))
    fig.write_html(FIGURE_DIR / f'{nama}.html', include_plotlyjs='directory',
                   full_html=True, config={'displaylogo': False})
    figures[nama] = fig
    fig.show(config={'displaylogo': False})

# Lazy scan dan proyeksi kolom: tidak perlu membaca Description untuk EDA ini.
lf = pl.scan_parquet(DATA_PATH)
required = {'Invoice', 'StockCode', 'Quantity', 'InvoiceDate', 'Price',
            'Price_Original', 'Country', 'Customer ID'}
missing = required - set(lf.collect_schema().names())
if missing:
    raise ValueError(f'Kolom hasil cleaning belum lengkap: {sorted(missing)}')
df = (
    lf.select(sorted(required))
    .with_columns(
        pl.col('Invoice').cast(pl.String).str.strip_chars(),
        pl.col('Quantity').cast(pl.Float64, strict=False),
        pl.col('Price_Original').cast(pl.Float64, strict=False),
        pl.col('InvoiceDate').cast(pl.Datetime('ms'), strict=False),
        pl.col('Country').cast(pl.String).str.strip_chars().fill_null('Unknown'),
    )
    .with_columns(
        (pl.col('Quantity') * pl.col('Price_Original')).alias('NilaiBaris'),
        pl.col('Invoice').str.to_uppercase().str.starts_with('C').alias('InvoiceBatal'),
        pl.col('InvoiceDate').dt.date().alias('Tanggal'),
    )
    .with_columns(
        pl.when(
            pl.col('InvoiceDate').is_null() | pl.col('Invoice').is_null()
            | pl.col('Invoice').str.to_lowercase().is_in(['', 'unknown'])
            | ~pl.col('Quantity').is_finite().fill_null(False)
            | ~pl.col('Price_Original').is_finite().fill_null(False)
            | ~pl.col('NilaiBaris').is_finite().fill_null(False)
        ).then(pl.lit('Tidak valid'))
        .when((pl.col('Price_Original') <= 0) | (pl.col('Quantity') == 0))
        .then(pl.lit('Harga nonpositif / kuantitas nol'))
        .when(pl.col('InvoiceBatal') | (pl.col('Quantity') < 0))
        .then(pl.lit('Retur / pembatalan'))
        .otherwise(pl.lit('Penjualan positif')).alias('Status')
    ).collect()
)
assert df.height > 0, 'Data bersih kosong.'
sales = df.filter(pl.col('Status') == 'Penjualan positif')
returns = df.filter(pl.col('Status') == 'Retur / pembatalan')
assert sales.height > 0, 'Tidak ada penjualan positif.'
audit = (
    df.group_by('Status').agg(pl.len().alias('JumlahBaris'))
    .with_columns((pl.col('JumlahBaris') / df.height * 100).alias('PersenBaris'))
    .sort('JumlahBaris', descending=True)
)
start_date, end_date = df['Tanggal'].min(), df['Tanggal'].max()
gross_sales = sales['NilaiBaris'].sum()
print(f'Polars {pl.__version__} | DuckDB {duckdb.__version__} | Plotly {plotly.__version__}')
print(f'Input: {df.height:,} baris | Penjualan: {sales.height:,} baris')
print(f'Periode: {start_date} sampai {end_date}')
print(f'Penjualan positif: GBP {gross_sales:,.2f}')
print(f'Harga yang dibatasi saat cleaning: {df.filter(pl.col("Price") != pl.col("Price_Original")).height:,} baris')
display(audit)

Polars 1.42.0 | DuckDB 2.0.0 | Plotly 7.1.0
Input: 1,033,036 baris | Penjualan: 1,007,913 baris
Periode: 2009-12-01 sampai 2011-12-09
Penjualan positif: GBP 20,476,260.45
Harga yang dibatasi saat cleaning: 66,191 baris


Status,JumlahBaris,PersenBaris
str,u32,f64
"""Penjualan positif""",1007913,97.568042
"""Retur / pembatalan""",19104,1.849306
"""Harga nonpositif / kuantitas n…",6019,0.582652


## Bagian 2: Analisis temporal dan window functions

`group_by_dynamic()` mengagregasi data yang telah diurutkan menjadi penjualan
harian dan bulanan. Kalender harian lengkap ditambahkan sebelum `rolling_mean(7)`
agar jendela benar-benar mencakup tujuh hari kalender. Enam nilai awal MA7 dibiarkan null.
DuckDB `LAG(...) OVER (ORDER BY Bulan)` membandingkan penjualan dengan bulan sebelumnya.
Pertumbuhan hanya dihitung ketika kedua bulan memiliki cakupan kalender penuh.

Agregasi invoice dan negara memakai seluruh penjualan yang memenuhi aturan di atas,
tanpa sampling. Rata-rata nilai invoice adalah total penjualan dibagi jumlah invoice.

In [ ]:
daily_raw = (
    sales.sort('InvoiceDate').group_by_dynamic('InvoiceDate', every='1d')
    .agg(pl.col('NilaiBaris').sum().alias('PenjualanGBP'),
         pl.col('Invoice').n_unique().alias('JumlahInvoice'),
         pl.col('Quantity').sum().alias('UnitTerjual'))
    .with_columns(pl.col('InvoiceDate').dt.date().alias('Tanggal')).drop('InvoiceDate')
)
calendar_days = pl.DataFrame({
    'Tanggal': pl.date_range(start_date, end_date, interval='1d', eager=True)
})
daily = (
    calendar_days.join(daily_raw, on='Tanggal', how='left')
    .with_columns(pl.col('PenjualanGBP', 'JumlahInvoice', 'UnitTerjual').fill_null(0))
    .sort('Tanggal')
    .with_columns(pl.col('PenjualanGBP').rolling_mean(window_size=7).alias('MA7_GBP'))
)
monthly_raw = (
    sales.sort('InvoiceDate').group_by_dynamic('InvoiceDate', every='1mo')
    .agg(pl.col('NilaiBaris').sum().alias('PenjualanGBP'),
         pl.col('Invoice').n_unique().alias('JumlahInvoice'))
    .with_columns(pl.col('InvoiceDate').dt.date().alias('Bulan')).drop('InvoiceDate')
)
month_calendar = pl.DataFrame({'Bulan': pl.date_range(
    start_date.replace(day=1), end_date.replace(day=1), interval='1mo', eager=True
)})
monthly_raw = (
    month_calendar.join(monthly_raw, on='Bulan', how='left')
    .with_columns(pl.col('PenjualanGBP', 'JumlahInvoice').fill_null(0))
    .with_columns(((pl.col('Bulan') >= pl.lit(start_date))
                   & (pl.col('Bulan').dt.month_end() <= pl.lit(end_date))).alias('BulanLengkap'))
    .sort('Bulan')
)
with duckdb.connect() as conn:
    conn.register('bulanan', monthly_raw.to_arrow())
    monthly = pl.from_arrow(conn.sql("""
        WITH lagged AS (
            SELECT *,
                LAG(PenjualanGBP) OVER (ORDER BY Bulan) AS PenjualanSebelumnya,
                LAG(BulanLengkap) OVER (ORDER BY Bulan) AS SebelumnyaLengkap
            FROM bulanan
        )
        SELECT *, CASE WHEN BulanLengkap AND SebelumnyaLengkap
            THEN 100.0 * (PenjualanGBP / NULLIF(PenjualanSebelumnya, 0) - 1)
            ELSE NULL END AS PertumbuhanMoM_Persen
        FROM lagged ORDER BY Bulan
    """).arrow())

invoices = (
    sales.group_by('Invoice').agg(
        pl.col('Country').first(), pl.col('Country').n_unique().alias('JumlahNegara'),
        pl.col('NilaiBaris').sum().alias('NilaiInvoice'),
        pl.col('Quantity').sum().alias('Unit'), pl.len().alias('JumlahBaris'),
    )
    .with_columns((pl.col('NilaiInvoice') / pl.col('Unit')).alias('HargaRataTertimbang'),
                  pl.col('NilaiInvoice').log10().alias('Log10Nilai'))
    .sort('Invoice')
)
assert invoices['JumlahNegara'].max() == 1, 'Periksa invoice yang terkait beberapa negara.'
country = (
    invoices.group_by('Country').agg(
        pl.col('NilaiInvoice').sum().alias('PenjualanGBP'),
        pl.len().alias('JumlahInvoice'), pl.col('Unit').sum().alias('UnitTerjual'),
    )
    .with_columns(
        (pl.col('PenjualanGBP') / pl.col('JumlahInvoice')).alias('RataRataInvoiceGBP'),
        (pl.col('PenjualanGBP') / gross_sales * 100).alias('PangsaPersen'),
        pl.when(pl.col('Country') == 'United Kingdom').then(pl.lit('United Kingdom'))
        .otherwise(pl.lit('Negara lainnya')).alias('Segmen'),
    ).sort('PenjualanGBP', descending=True)
)
display(monthly)
display(country.head(10))

Bulan,PenjualanGBP,JumlahInvoice,BulanLengkap,PenjualanSebelumnya,SebelumnyaLengkap,PertumbuhanMoM_Persen
date,f64,u32,bool,f64,bool,f64
2009-12-01,822483.95,1682,true,null,null,null
2010-01-01,651155.112,1105,true,822483.95,true,-20.83066
2010-02-01,551504.726,1201,true,651155.112,true,-15.303633
2010-03-01,830915.261,1681,true,551504.726,true,50.663307
2010-04-01,678875.252,1462,true,830915.261,true,-18.297896
…,…,…,…,…,…,…
2011-08-01,757841.38,1361,true,718076.121,true,5.53775
2011-09-01,1.0564e6,1837,true,757841.38,true,39.400568
2011-10-01,1.1513e6,2040,true,1.0564e6,true,8.976276


Country,PenjualanGBP,JumlahInvoice,UnitTerjual,RataRataInvoiceGBP,PangsaPersen,Segmen
str,f64,u32,f64,f64,f64,str
"""United Kingdom""",1.7410e7,36535,9.187753e6,476.534723,85.026249,"""United Kingdom"""
"""EIRE""",658767.31,626,336329.0,1052.343946,3.217225,"""Negara lainnya"""
"""Netherlands""",554038.09,228,383879.0,2429.991623,2.705758,"""Negara lainnya"""
"""Germany""",425019.711,789,225154.0,538.68151,2.075671,"""Negara lainnya"""
"""France""",350456.09,622,271901.0,563.434228,1.711524,"""Negara lainnya"""
"""Australia""",169283.46,95,103759.0,1781.931158,0.82673,"""Negara lainnya"""
"""Spain""",108332.49,154,50307.0,703.457727,0.529064,"""Negara lainnya"""
"""Switzerland""",100685.59,93,52762.0,1082.640753,0.491719,"""Negara lainnya"""
"""Sweden""",91869.82,105,88633.0,874.950667,0.448665,"""Negara lainnya"""


## Bagian 3: Enam visualisasi interaktif

Semua grafik memiliki hover dan zoom serta disimpan sebagai HTML di
`output/figures/`. Pertahankan `plotly.min.js` dalam folder yang sama agar HTML
bisa dibuka tanpa internet.

1. **Tren:** penjualan harian dan MA7 dengan range slider; hari terakhir bisa parsial.
2. **Distribusi:** histogram log10 nilai invoice tanpa menghapus outlier;
   log10 = 1 berarti GBP 10, 2 berarti GBP 100, dan 3 berarti GBP 1.000.
3. **Korelasi:** Pearson pada tingkat invoice. Kuantitas dan harga membentuk nilai
   invoice, sehingga hubungan bersifat deskriptif, bukan bukti sebab akibat.
4. **Komposisi:** 10 negara terbesar dan kategori Lainnya, mencakup seluruh penjualan.
5. **Siklus mingguan:** penjualan tiap kombinasi hari–jam dibagi jumlah hari kalender
   dengan nama hari tersebut. Hari terakhir dikeluarkan karena parsial. Sel nol
   berarti tidak ada penjualan teramati pada kombinasi itu.
6. **Multidimensi:** jumlah invoice, nilai rata-rata invoice, dan total penjualan per
   negara. Negara dengan sedikit invoice dapat memiliki rata-rata yang tidak stabil.

In [ ]:
# Visualisasi 1: tren penjualan dan rata-rata bergerak 7 hari.
fig1 = go.Figure()
fig1.add_scatter(x=daily['Tanggal'].to_list(), y=daily['PenjualanGBP'].to_list(),
                 name='Penjualan harian', mode='lines', line=dict(color='#8AB6D6', width=1),
                 hovertemplate='%{x|%d %b %Y}<br>GBP %{y:,.2f}<extra>Harian</extra>')
fig1.add_scatter(x=daily['Tanggal'].to_list(), y=daily['MA7_GBP'].to_list(),
                 name='Rata-rata 7 hari', mode='lines', line=dict(color='#0072B2', width=2.5),
                 hovertemplate='%{x|%d %b %Y}<br>MA7 GBP %{y:,.2f}<extra></extra>')
fig1.update_layout(title='Tren penjualan positif harian dan MA7',
                   xaxis_title='Tanggal invoice', yaxis_title='Penjualan (GBP)',
                   hovermode='x unified', height=500, legend=dict(orientation='h', y=1.12))
fig1.update_xaxes(rangeslider_visible=True, rangeselector=dict(buttons=[
    dict(count=3, label='3 bulan', step='month', stepmode='backward'),
    dict(count=1, label='1 tahun', step='year', stepmode='backward'),
    dict(step='all', label='Semua'),
]))
tampilkan(fig1, '03_01_tren_harian')

In [ ]:
# Visualisasi 2: seluruh invoice, termasuk pesanan ekstrem.
median_invoice = invoices['NilaiInvoice'].median()
p99_invoice = invoices['NilaiInvoice'].quantile(0.99, interpolation='linear')
fig2 = px.histogram(invoices.select('Log10Nilai'), x='Log10Nilai', nbins=70,
                    labels={'Log10Nilai': 'Log10 nilai invoice (GBP)'},
                    title='Distribusi nilai pesanan: seluruh invoice penjualan positif')
fig2.add_vline(x=math.log10(median_invoice), line_dash='dash', line_color='#009E73',
               annotation_text=f'Median GBP {median_invoice:,.2f}', annotation_position='top left')
fig2.add_vline(x=math.log10(p99_invoice), line_dash='dot', line_color='#D55E00',
               annotation_text=f'P99 GBP {p99_invoice:,.2f}', annotation_position='top right')
fig2.update_layout(yaxis_title='Jumlah invoice', bargap=0.04, height=470)
tampilkan(fig2, '03_02_distribusi_invoice')

In [ ]:
# Visualisasi 3: Pearson dihitung pada seluruh invoice, bukan sampel baris item.
numeric_cols = ['Unit', 'JumlahBaris', 'HargaRataTertimbang', 'NilaiInvoice']
corr_matrix = [[invoices.select(pl.corr(a, b)).item() for b in numeric_cols]
               for a in numeric_cols]
corr_labels = ['Total unit', 'Jumlah baris item', 'Harga rata-rata/unit', 'Nilai invoice']
fig3 = go.Figure(go.Heatmap(
    z=corr_matrix, x=corr_labels, y=corr_labels, zmin=-1, zmax=1, zmid=0,
    colorscale='RdBu_r', texttemplate='%{z:.2f}', colorbar=dict(title='Pearson r'),
    hovertemplate='%{y} vs %{x}<br>Pearson r = %{z:.3f}<extra></extra>',
))
fig3.update_layout(title='Korelasi karakteristik pesanan pada seluruh invoice', height=480)
tampilkan(fig3, '03_03_korelasi_invoice')

In [ ]:
# Visualisasi 4: komposisi kategori, dengan Lainnya agar total tetap lengkap.
top_country = country.head(10).select('Country', 'PenjualanGBP')
if country.height > 10:
    top_country = pl.concat([top_country, pl.DataFrame({
        'Country': ['Lainnya'], 'PenjualanGBP': [country.slice(10)['PenjualanGBP'].sum()]
    })])
top_country = top_country.with_columns(
    (pl.col('PenjualanGBP') / gross_sales * 100).alias('PangsaPersen')
).sort('PenjualanGBP')
fig4 = px.bar(top_country, x='PenjualanGBP', y='Country', orientation='h',
              text='PangsaPersen', custom_data=['PangsaPersen'],
              labels={'PenjualanGBP': 'Penjualan positif (GBP)', 'Country': 'Negara pelanggan'},
              title='Kontribusi 10 negara terbesar dan negara lainnya')
fig4.update_traces(texttemplate='%{text:.2f}%', textposition='outside', cliponaxis=False,
                   hovertemplate='%{y}<br>GBP %{x:,.2f}<br>Pangsa %{customdata[0]:.2f}%<extra></extra>')
fig4.update_layout(height=530, xaxis=dict(range=[0, top_country['PenjualanGBP'].max() * 1.15]))
tampilkan(fig4, '03_04_komposisi_negara')

In [ ]:
# Visualisasi 5: normalisasi dengan jumlah hari kalender, tanpa hari terakhir.
day_names = ['Senin', 'Selasa', 'Rabu', 'Kamis', 'Jumat', 'Sabtu', 'Minggu']
day_counts = (
    calendar_days.filter(pl.col('Tanggal') < pl.lit(end_date))
    .with_columns(pl.col('Tanggal').dt.weekday().alias('HariKe'))
    .group_by('HariKe').len(name='JumlahHariKalender')
)
weekday_hour_raw = (
    sales.filter(pl.col('Tanggal') < pl.lit(end_date))
    .with_columns(pl.col('InvoiceDate').dt.weekday().alias('HariKe'),
                  pl.col('InvoiceDate').dt.hour().alias('Jam'))
    .group_by('HariKe', 'Jam').agg(pl.col('NilaiBaris').sum().alias('PenjualanGBP'))
)
weekday_hour = (
    pl.DataFrame({'HariKe': range(1, 8)}).join(pl.DataFrame({'Jam': range(24)}), how='cross')
    .join(weekday_hour_raw, on=['HariKe', 'Jam'], how='left')
    .join(day_counts, on='HariKe', how='left')
    .with_columns(pl.col('PenjualanGBP').fill_null(0))
    .with_columns((pl.col('PenjualanGBP') / pl.col('JumlahHariKalender')).alias('RataRataGBP'))
    .sort('HariKe', 'Jam')
)
hour_matrix = [weekday_hour.filter(pl.col('HariKe') == i)['RataRataGBP'].to_list()
               for i in range(1, 8)]
fig5 = go.Figure(go.Heatmap(
    z=hour_matrix, x=list(range(24)), y=day_names, colorscale='YlGnBu',
    colorbar=dict(title='GBP/hari'),
    hovertemplate='%{y}, jam %{x}:00<br>Rata-rata GBP %{z:,.2f}/hari<extra></extra>',
))
fig5.update_layout(title='Pola mingguan: rata-rata penjualan pada setiap hari dan jam',
                   xaxis_title='Jam invoice (waktu sumber)', yaxis_title='Hari', height=460)
fig5.update_xaxes(dtick=1)
fig5.update_yaxes(autorange='reversed')
tampilkan(fig5, '03_05_pola_hari_jam')

In [ ]:
# Visualisasi 6: analisis multidimensi negara dengan sumbu logaritmik.
fig6 = px.scatter(
    country, x='JumlahInvoice', y='RataRataInvoiceGBP', size='PenjualanGBP',
    color='Segmen', hover_name='Country', log_x=True, log_y=True, size_max=65,
    hover_data={'PenjualanGBP': ':,.2f', 'PangsaPersen': ':.2f',
                'JumlahInvoice': ':,', 'RataRataInvoiceGBP': ':,.2f', 'Segmen': False},
    labels={'JumlahInvoice': 'Jumlah invoice (log)',
            'RataRataInvoiceGBP': 'Rata-rata nilai invoice, GBP (log)',
            'PenjualanGBP': 'Penjualan (GBP)', 'PangsaPersen': 'Pangsa (%)'},
    title='Profil pasar: volume invoice, nilai rata-rata, dan total penjualan',
)
fig6.update_layout(height=530, legend=dict(title='Kelompok negara', orientation='h', y=-0.25))
tampilkan(fig6, '03_06_profil_negara')

## Bagian 4: Enam insight analitik utama

Angka berikut dihitung ulang dari data pada setiap eksekusi. Pertumbuhan tahunan
membandingkan Januari–November 2010 dan 2011 agar periode setara. Interpretasi
berikut bersifat deskriptif terhadap data historis, bukan eksperimen atau prediksi.

In [ ]:
full_months = monthly.filter(pl.col('BulanLengkap'))
matched = (
    full_months.with_columns(pl.col('Bulan').dt.year().alias('Tahun'),
                             pl.col('Bulan').dt.month().alias('BulanKe'))
    .filter(pl.col('BulanKe') <= 11)
)
for year in (2010, 2011):
    assert matched.filter(pl.col('Tahun') == year).height == 11, 'Periode YoY belum lengkap.'
revenue_2010 = matched.filter(pl.col('Tahun') == 2010)['PenjualanGBP'].sum()
revenue_2011 = matched.filter(pl.col('Tahun') == 2011)['PenjualanGBP'].sum()
yoy = (revenue_2011 / revenue_2010 - 1) * 100
peak_month = full_months.sort('PenjualanGBP', descending=True).row(0, named=True)
top_n = max(1, math.ceil(invoices.height * 0.01))
top1_share = invoices.sort('NilaiInvoice', descending=True).head(top_n)['NilaiInvoice'].sum() / gross_sales * 100
unit_value_r = invoices.select(pl.corr('Unit', 'NilaiInvoice')).item()
line_value_r = invoices.select(pl.corr('JumlahBaris', 'NilaiInvoice')).item()
leader, runner_up = country.row(0, named=True), country.row(1, named=True)
peak_slot = weekday_hour.sort('RataRataGBP', descending=True).row(0, named=True)
peak_day = (
    weekday_hour.group_by('HariKe').agg(pl.col('RataRataGBP').sum().alias('RataRataHarianGBP'))
    .sort('RataRataHarianGBP', descending=True).row(0, named=True)
)
missing_customer_share = sales['Customer ID'].null_count() / sales.height * 100
return_abs_value = returns['NilaiBaris'].abs().sum()
direction = 'naik' if yoy >= 0 else 'turun'
insights = f"""
1. **Tren waktu dan musiman (Grafik 1, tabel bulanan).** Penjualan Januari–November
   2011 sebesar **GBP {revenue_2011:,.2f}**, dibanding **GBP {revenue_2010:,.2f}**
   pada periode sama tahun 2010 ({direction} **{abs(yoy):.2f}%**). Bulan penuh
   tertinggi adalah **{peak_month['Bulan']:%Y-%m}**, sebesar
   **GBP {peak_month['PenjualanGBP']:,.2f}**. Periode tersebut menjadi masukan awal
   persiapan stok. Dua tahun observasi belum cukup untuk memastikan pola musiman;
   Desember 2011 yang parsial tidak digunakan untuk menyimpulkan penurunan bulanan.

2. **Distribusi pesanan tidak merata (Grafik 2).** Median invoice adalah
   **GBP {median_invoice:,.2f}**, sedangkan P99 mencapai **GBP {p99_invoice:,.2f}**.
   Sebanyak **{top_n:,} invoice terbesar** (sekitar 1% dari {invoices.height:,}
   invoice) menyumbang **{top1_share:.2f}%** penjualan positif. Gunakan median
   untuk pesanan tipikal dan tinjau pesanan ekstrem yang dapat mengangkat rata-rata.

3. **Hubungan ukuran keranjang dengan nilai pesanan (Grafik 3).** Korelasi total
   unit dengan nilai invoice adalah **r = {unit_value_r:.3f}**, sedangkan jumlah
   baris item dengan nilai invoice **r = {line_value_r:.3f}**. Keduanya mengukur
   aspek berbeda dari ukuran pesanan. Korelasi juga dipengaruhi konstruksi
   kuantitas × harga serta outlier; ini tidak membuktikan efek kausal bundling atau diskon.

4. **Kontribusi pasar terkonsentrasi (Grafik 4 dan 6).** **{leader['Country']}**
   menyumbang **{leader['PangsaPersen']:.2f}%** penjualan dari
   **{leader['JumlahInvoice']:,} invoice**, rata-rata
   **GBP {leader['RataRataInvoiceGBP']:,.2f}** per invoice. Peringkat berikutnya,
   **{runner_up['Country']}**, menyumbang **{runner_up['PangsaPersen']:.2f}%**
   dengan rata-rata **GBP {runner_up['RataRataInvoiceGBP']:,.2f}** per invoice.
   Prioritaskan layanan pasar utama; nilai peluang ekspansi memakai volume dan
   nilai pesanan bersama-sama, kemudian validasi biaya pengiriman dan margin.

5. **Penjadwalan operasional (Grafik 5).** Rata-rata harian tertinggi terjadi pada
   **{day_names[peak_day['HariKe'] - 1]}**, sebesar
   **GBP {peak_day['RataRataHarianGBP']:,.2f} per hari kalender**. Slot paling
   intensif adalah **{day_names[peak_slot['HariKe'] - 1]}, {peak_slot['Jam']:02d}:00**,
   rata-rata **GBP {peak_slot['RataRataGBP']:,.2f}** pada jam tersebut. Jadwal
   pemrosesan pesanan dapat memakai pola ini sebagai masukan; waktu invoice
   tidak mengukur kunjungan pelanggan, waktu pengiriman, atau durasi kerja gudang.

6. **Batas cakupan metrik (audit Bagian 1).** **{returns.height:,} baris
   retur/pembatalan** bernilai absolut **GBP {return_abs_value:,.2f}** dipisahkan
   dari penjualan positif karena belum dipasangkan ke invoice asal. Sebanyak
   **{missing_customer_share:.2f}%** baris penjualan tidak memiliki ID pelanggan.
   Total penjualan bisa dianalisis, tetapi cakupan analisis loyalitas lebih kecil.
   Total penjualan positif tidak dapat langsung disebut pendapatan neto atau laba.
"""
display(Markdown(insights))


1. **Tren waktu dan musiman (Grafik 1, tabel bulanan).** Penjualan Januari–November
   2011 sebesar **GBP 9,182,867.74**, dibanding **GBP 9,011,647.69**
   pada periode sama tahun 2010 (naik **1.90%**). Bulan penuh
   tertinggi adalah **2011-11**, sebesar
   **GBP 1,503,866.78**. Periode tersebut menjadi masukan awal
   persiapan stok. Dua tahun observasi belum cukup untuk memastikan pola musiman;
   Desember 2011 yang parsial tidak digunakan untuk menyimpulkan penurunan bulanan.

2. **Distribusi pesanan tidak merata (Grafik 2).** Median invoice adalah
   **GBP 302.22**, sedangkan P99 mencapai **GBP 4,408.59**.
   Sebanyak **401 invoice terbesar** (sekitar 1% dari 40,077
   invoice) menyumbang **18.02%** penjualan positif. Gunakan median
   untuk pesanan tipikal dan tinjau pesanan ekstrem yang dapat mengangkat rata-rata.

3. **Hubungan ukuran keranjang dengan nilai pesanan (Grafik 3).** Korelasi total
   unit dengan nilai invoice adalah **r = 0.628**, sedangkan jumlah
   baris item dengan nilai invoice **r = 0.284**. Keduanya mengukur
   aspek berbeda dari ukuran pesanan. Korelasi juga dipengaruhi konstruksi
   kuantitas × harga serta outlier; ini tidak membuktikan efek kausal bundling atau diskon.

4. **Kontribusi pasar terkonsentrasi (Grafik 4 dan 6).** **United Kingdom**
   menyumbang **85.03%** penjualan dari
   **36,535 invoice**, rata-rata
   **GBP 476.53** per invoice. Peringkat berikutnya,
   **EIRE**, menyumbang **3.22%**
   dengan rata-rata **GBP 1,052.34** per invoice.
   Prioritaskan layanan pasar utama; nilai peluang ekspansi memakai volume dan
   nilai pesanan bersama-sama, kemudian validasi biaya pengiriman dan margin.

5. **Penjadwalan operasional (Grafik 5).** Rata-rata harian tertinggi terjadi pada
   **Kamis**, sebesar
   **GBP 39,577.48 per hari kalender**. Slot paling
   intensif adalah **Rabu, 12:00**,
   rata-rata **GBP 5,320.07** pada jam tersebut. Jadwal
   pemrosesan pesanan dapat memakai pola ini sebagai masukan; waktu invoice
   tidak mengukur kunjungan pelanggan, waktu pengiriman, atau durasi kerja gudang.

6. **Batas cakupan metrik (audit Bagian 1).** **19,104 baris
   retur/pembatalan** bernilai absolut **GBP 1,462,797.75** dipisahkan
   dari penjualan positif karena belum dipasangkan ke invoice asal. Sebanyak
   **22.67%** baris penjualan tidak memiliki ID pelanggan.
   Total penjualan bisa dianalisis, tetapi cakupan analisis loyalitas lebih kecil.
   Total penjualan positif tidak dapat langsung disebut pendapatan neto atau laba.


## Bagian 5: Validasi dan simpan hasil

Rekonsiliasi memeriksa bahwa total penjualan konsisten pada agregasi harian,
bulanan, invoice, dan negara. Semua agregat dapat dibuat ulang; data sumber tidak diubah.

In [ ]:
assert audit['JumlahBaris'].sum() == df.height
assert daily.height == (end_date - start_date).days + 1
assert daily['MA7_GBP'].head(6).null_count() == 6
assert math.isclose(daily['MA7_GBP'][6], daily['PenjualanGBP'].head(7).mean(), rel_tol=1e-9)
assert sales['NilaiBaris'].min() > 0
assert invoices.height == sales['Invoice'].n_unique()
for name, total in {
    'harian': daily['PenjualanGBP'].sum(), 'bulanan': monthly['PenjualanGBP'].sum(),
    'invoice': invoices['NilaiInvoice'].sum(), 'negara': country['PenjualanGBP'].sum(),
    'komposisi': top_country['PenjualanGBP'].sum(),
}.items():
    assert math.isclose(total, gross_sales, rel_tol=1e-9, abs_tol=0.01), name
assert len(figures) == 6
assert all((FIGURE_DIR / f'{name}.html').is_file() for name in figures)
assert (FIGURE_DIR / 'plotly.min.js').is_file()
partial = monthly.filter(~pl.col('BulanLengkap'))
assert partial['PertumbuhanMoM_Persen'].null_count() == partial.height
for name, table in {'audit': audit, 'harian': daily, 'bulanan': monthly,
                    'negara': country, 'hari_jam': weekday_hour}.items():
    table.write_csv(TABLE_DIR / f'03_{name}.csv')
(TABLE_DIR / '03_insights.md').write_text(insights.strip() + chr(10), encoding='utf-8')
print('VALIDASI BERHASIL: audit, MA7, rekonsiliasi total, dan enam grafik interaktif.')
print('Grafik: output/figures/03_*.html | Agregat: output/03_*.csv')

VALIDASI BERHASIL: audit, MA7, rekonsiliasi total, dan enam grafik interaktif.
Grafik: output/figures/03_*.html | Agregat: output/03_*.csv
